In [ ]:
import argparse
import os
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader
from torch.utils.data import TensorDataset


In [ ]:



def cartesian_heart_dataset(n=8000):
    rng = np.random.default_rng(42)
    theta = 2 * np.pi * rng.uniform(0, 1, n)
    x = (1 - np.sin(theta)) * np.cos(theta)
    y = (1 - np.sin(theta)) * np.sin(theta) + 0.9
    X = np.stack((x, y), axis=1)
    X *= 3
    return TensorDataset(torch.from_numpy(X.astype(np.float32)))

def perfect_heart_dataset(n=8000):
    rng = np.random.default_rng(42)
    theta = 2 * np.pi * rng.uniform(0, 1, n)
    x = 16 * np.sin(theta)**3
    y = 13 * np.cos(theta) - 5 * np.cos(2*theta) - 2 * np.cos(3*theta) - np.cos(4*theta)
    X = np.stack((x, y), axis=1)
    X /= 5
    # Repeat features to get 147 dimensions
    X = np.tile(X, (1, 74))  # 2 * 74 = 148
    X = X[:, :147]          # Trim to 147

    data = torch.from_numpy(X.astype(np.float32))
    print("the input is: ",data.shape)
    return TensorDataset(data)

def mnist_dataset(train=True):
    from torchvision.datasets import MNIST
    from torchvision import transforms
    transform = transforms.Compose([transforms.ToTensor()])
    train_dataset = MNIST(
            "./data", download=True, train=train, transform=transform
        )
    data = train_dataset.data
    data = ((data / 255.0) * 2.0) - 1.0
    data = data.reshape(-1, 28*28)
    return data

def get_dataset(name, n=8000):
    if name == "heart":
        return perfect_heart_dataset(n)
    elif name == "mnist":
        return mnist_dataset()
    else:
        raise ValueError(f"Unknown dataset: {name}")

class Block(nn.Module):
    def __init__(self, size: int):
        super().__init__()
        self.ln = nn.LayerNorm(size)
        self.ff = nn.Linear(size, size)
        self.act = nn.GELU()

    def forward(self, x: torch.Tensor):
        return x + self.act(self.ff(self.ln(x)))

class SinusoidalEmbedding(nn.Module):
    def __init__(self, size: int, scale: float = 1.0):
        super().__init__()
        self.size = size
        self.scale = scale
        half_size = self.size // 2
        emb = torch.log(torch.Tensor([10000.0])) / (half_size - 1)
        emb = torch.exp(-emb * torch.arange(half_size))
        self.emb = nn.Parameter(emb, requires_grad=False)

    def forward(self, x: torch.Tensor):
        x = x * self.scale
        emb = x * self.emb.unsqueeze(0)
        emb = torch.cat((torch.sin(emb), torch.cos(emb)), dim=-1)
        return emb


class MLP(nn.Module):
    def __init__(self, data_dim=2, hidden_size=128, hidden_layers=3, emb_size=128):
        super().__init__()
        self.data_dim = data_dim

        self.time_emb = SinusoidalEmbedding(emb_size)

        self.concat_size = data_dim + emb_size
        self.data_size = data_dim

        layers = [nn.Linear(self.concat_size, hidden_size), nn.GELU()]
        for _ in range(hidden_layers):
            layers.append(Block(hidden_size))
        layers.append(nn.LayerNorm(hidden_size))
        layers.append(nn.Linear(hidden_size, self.data_size))

        self.joint_mlp = nn.Sequential(*layers)

    def forward(self, x, t):
        t_emb = self.time_emb(t.reshape(-1,1))
        x = torch.cat((x, t_emb), dim=-1)
        return self.joint_mlp(x)

class Diffusion():
    def __init__(self,
                 num_timesteps=1000,
                 beta_start=0.0001,
                 beta_end=0.02):

        self.num_timesteps = num_timesteps
        self.betas = torch.linspace(
                beta_start, beta_end, num_timesteps, dtype=torch.float32)

        self.alphas = 1.0 - self.betas
        self.alphas_cumprod = torch.cumprod(self.alphas, axis=0)
        self.alphas_cumprod_prev = F.pad(
            self.alphas_cumprod[:-1], (1, 0), value=1.)

        self.sqrt_alphas_cumprod = self.alphas_cumprod ** 0.5
        self.sqrt_one_minus_alphas_cumprod = (1 - self.alphas_cumprod) ** 0.5

        self.sqrt_inv_alphas = torch.sqrt(1. / self.alphas)
        self.noise_coef = self.betas / self.sqrt_one_minus_alphas_cumprod
        self.variance = self.betas * (1. - self.alphas_cumprod_prev) / (1. - self.alphas_cumprod)

    def add_noise(self, x_start, x_noise, timesteps):
        # compute x_t for DDPM algorithm 1
        s1 = self.sqrt_alphas_cumprod[timesteps].reshape(-1, 1)
        s2 = self.sqrt_one_minus_alphas_cumprod[timesteps].reshape(-1, 1)
        return s1 * x_start + s2 * x_noise
    
    def sample_step(self, model_output, timestep, sample):
        # compute x_{t-1} for DDPM algorithm 2
        s1 = self.sqrt_inv_alphas[timestep].reshape(-1, 1)
        s2 = self.noise_coef[timestep].reshape(-1, 1)
        s3 = self.variance[timestep].reshape(-1, 1) ** 0.5
        noise = torch.randn_like(model_output)
        return s1 * (sample - s2 * model_output) + s3 * noise

    def __len__(self):
        return self.num_timesteps



In [26]:
experiment_name = "base"
dataset = "heart"
train_batch_size = 32
eval_batch_size= 1000
num_epochs= 1
learning_rate= 1e-3
num_timesteps= 200
embedding_size= 32
hidden_size= 32
hidden_layers= 1
eval_path= None
device= "cpu"

outdir = f"exps/{experiment_name}"

In [27]:
if device[:4] =="cuda":
    device = device
else:
    device = device = "cpu"

dataset = get_dataset(dataset)
dataloader = DataLoader(
    dataset, batch_size=train_batch_size, shuffle=True, drop_last=True)

model = MLP(
    data_dim=147,
    hidden_size=hidden_size,
    hidden_layers=hidden_layers,
    emb_size=embedding_size).to(device)

diffusion = Diffusion(
    num_timesteps=num_timesteps)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate,
)

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda step: 1 - step / (num_epochs * len(dataloader)))

the input is:  torch.Size([8000, 147])


In [28]:
if eval_path is None:
    global_step = 0
    losses = []
    print("Training model...")
    for epoch in range(num_epochs):
        model.train()
        progress_bar = tqdm(total=len(dataloader))
        progress_bar.set_description(f"Epoch {epoch}")
        for step, batch in enumerate(dataloader):
            if type(batch) == list:
                batch = batch[0]
            noise = torch.randn(batch.shape)
            timesteps = torch.randint(
                0, diffusion.num_timesteps, (batch.shape[0],)
            ).long()

            noisy = diffusion.add_noise(batch, noise, timesteps)
            noise_pred = model(noisy.to(device), timesteps.to(device))
            loss = F.mse_loss(noise_pred, noise.to(device))
            loss.backward()

            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad()
            scheduler.step()

            progress_bar.update(1)
            logs = {"loss": loss.detach().item(), "step": global_step, "lr": scheduler.get_last_lr()[0]}
            losses.append(loss.detach().item())
            progress_bar.set_postfix(**logs)
            global_step += 1
        progress_bar.close()

    print("Saving model...")
    os.makedirs(outdir, exist_ok=True)
    torch.save(model.state_dict(), f"{outdir}/model.pth")

Training model...


  0%|          | 0/250 [00:00<?, ?it/s]

Saving model...


In [3]:
n= 8000
rng = np.random.default_rng(42)
theta = 2 * np.pi * rng.uniform(0, 1, n)
x = 16 * np.sin(theta)**3
y = 13 * np.cos(theta) - 5 * np.cos(2*theta) - 2 * np.cos(3*theta) - np.cos(4*theta)
X = np.stack((x, y), axis=1)
X /= 5
data = torch.from_numpy(X.astype(np.float32))
print(data.shape)

torch.Size([8000, 2])


In [29]:
type(data)

torch.Tensor